## This step tests the feedback loop: does housing price movement predict macro variables, and do macro variables predict housing prices? Then we test whether this relationship is regime-dependent.

In [3]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Load the merged Manhattan index
SAVE_DIR = "/Users/alicexu/Documents/MSAA/2026_Spring/5205/AML2_Final_Project/Dataset/Section 2 -Housing Market Heterogeneity"
MACRO_PATH = "/Users/alicexu/Documents/MSAA/2026_Spring/5205/AML2_Final_Project/Dataset/macro_regimes_final.csv"

manhattan_index = pd.read_csv(f"{SAVE_DIR}/manhattan_regime_merged.csv",
                              parse_dates=["Unnamed: 0"])
manhattan_index = manhattan_index.rename(columns={"Unnamed: 0": "month"})
manhattan_index = manhattan_index.set_index("month").sort_index()

# Build the Granger dataset
granger_df = manhattan_index[["appreciation", "FEDFUNDS", "UNRATE",
                               "PCE_YOY", "MORTGAGE_SPREAD", "regime_name"]].copy()

print(f"Shape: {granger_df.shape}")
print(f"Date range: {granger_df.index.min()} → {granger_df.index.max()}")
print(f"Null counts:\n{granger_df.isnull().sum()}")

Shape: (115, 6)
Date range: 2016-02-01 00:00:00 → 2025-12-01 00:00:00
Null counts:
appreciation       0
FEDFUNDS           0
UNRATE             0
PCE_YOY            0
MORTGAGE_SPREAD    0
regime_name        0
dtype: int64


In [4]:
# stationary check
# identify non-stationary series
print("=" * 60)
print("ADF Stationarity Test")
print("=" * 60)

test_cols = ["appreciation", "FEDFUNDS", "UNRATE", "PCE_YOY", "MORTGAGE_SPREAD"]

stationarity = {}
for col in test_cols:
    result = adfuller(granger_df[col].dropna())
    pval = result[1]
    stationary = pval < 0.05
    stationarity[col] = stationary
    status = "Stationary" if stationary else "NON-STATIONARY → needs differencing"
    print(f"  {col:<20} p={pval:.4f}   {status}")

print("=" * 60)

ADF Stationarity Test
  appreciation         p=0.0000   Stationary
  FEDFUNDS             p=0.2092   NON-STATIONARY → needs differencing
  UNRATE               p=0.0158   Stationary
  PCE_YOY              p=0.2283   NON-STATIONARY → needs differencing
  MORTGAGE_SPREAD      p=0.0807   NON-STATIONARY → needs differencing


In [5]:
# Difference non-stationary series (change to monthly change instead)
diff_cols = [col for col, is_stat in stationarity.items() if not is_stat]

if diff_cols:
    print(f"\nDifferencing: {diff_cols}")
    for col in diff_cols:
        granger_df[f"d_{col}"] = granger_df[col].diff()

    granger_df = granger_df.dropna()

    # Build the final variable list for Granger testing
    var_cols = []
    for col in test_cols:
        if col in diff_cols:
            var_cols.append(f"d_{col}")
        else:
            var_cols.append(col)

    # Verify stationarity after differencing
    print("\nPost-differencing ADF check:")
    for col in var_cols:
        result = adfuller(granger_df[col].dropna())
        print(f"  {col:<20} p={result[1]:.4f}")
else:
    var_cols = test_cols
    print("\nAll series already stationary — no differencing needed")

print(f"\nFinal variables for Granger test: {var_cols}")
print(f"Observations: {len(granger_df)}")


Differencing: ['FEDFUNDS', 'PCE_YOY', 'MORTGAGE_SPREAD']

Post-differencing ADF check:
  appreciation         p=0.0000
  d_FEDFUNDS           p=0.0624
  UNRATE               p=0.0164
  d_PCE_YOY            p=0.0012
  d_MORTGAGE_SPREAD    p=0.0010

Final variables for Granger test: ['appreciation', 'd_FEDFUNDS', 'UNRATE', 'd_PCE_YOY', 'd_MORTGAGE_SPREAD']
Observations: 114


In [6]:
# Identify which column is appreciation (raw or differenced)
appr_col = "d_appreciation" if "d_appreciation" in var_cols else "appreciation"
macro_cols = [c for c in var_cols if c != appr_col]

max_lag = 6  # Test up to 6 months of lags
# We tested lags 1 through 6 (1 month back through 6 months back) because the housing market doesn't respond instantly — there's a delay between macro shifts and closed transaction prices.

print("=" * 75)
print("FULL-SAMPLE GRANGER CAUSALITY TESTS")
print(f"Max lags: {max_lag} | Appreciation variable: {appr_col}")
print("=" * 75)

results_list = []

for macro_var in macro_cols:
    # Direction 1: Macro → Housing
    print(f"\n{'─' * 75}")
    print(f"TEST: {macro_var} → {appr_col}")
    try:
        test1 = grangercausalitytests(
            granger_df[[appr_col, macro_var]].dropna(),
            maxlag=max_lag, verbose=False
        )
        for lag in range(1, max_lag + 1):
            f_stat = test1[lag][0]["ssr_ftest"][0]
            p_val  = test1[lag][0]["ssr_ftest"][1]
            sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.10 else ""
            results_list.append({
                "direction": f"{macro_var} → {appr_col}",
                "lag": lag, "F_stat": round(f_stat, 3),
                "p_value": round(p_val, 4), "sig": sig
            })
            if lag <= 3:  # Print first 3 lags
                print(f"  Lag {lag}: F={f_stat:>7.3f}  p={p_val:.4f}  {sig}")
    except Exception as e:
        print(f"  Error: {e}")

    # Direction 2: Housing → Macro
    print(f"TEST: {appr_col} → {macro_var}")
    try:
        test2 = grangercausalitytests(
            granger_df[[macro_var, appr_col]].dropna(),
            maxlag=max_lag, verbose=False
        )
        for lag in range(1, max_lag + 1):
            f_stat = test2[lag][0]["ssr_ftest"][0]
            p_val  = test2[lag][0]["ssr_ftest"][1]
            sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.10 else ""
            results_list.append({
                "direction": f"{appr_col} → {macro_var}",
                "lag": lag, "F_stat": round(f_stat, 3),
                "p_value": round(p_val, 4), "sig": sig
            })
            if lag <= 3:
                print(f"  Lag {lag}: F={f_stat:>7.3f}  p={p_val:.4f}  {sig}")
    except Exception as e:
        print(f"  Error: {e}")

print(f"\n{'=' * 75}")

# Summary table: best lag per direction
results_df = pd.DataFrame(results_list)

FULL-SAMPLE GRANGER CAUSALITY TESTS
Max lags: 6 | Appreciation variable: appreciation

───────────────────────────────────────────────────────────────────────────
TEST: d_FEDFUNDS → appreciation
  Lag 1: F=  0.210  p=0.6477  
  Lag 2: F=  0.161  p=0.8511  
  Lag 3: F=  0.141  p=0.9355  
TEST: appreciation → d_FEDFUNDS
  Lag 1: F=  3.368  p=0.0692  *
  Lag 2: F=  1.657  p=0.1955  
  Lag 3: F=  1.147  p=0.3338  

───────────────────────────────────────────────────────────────────────────
TEST: UNRATE → appreciation
  Lag 1: F=  0.859  p=0.3560  
  Lag 2: F=  0.669  p=0.5142  
  Lag 3: F=  0.485  p=0.6937  
TEST: appreciation → UNRATE
  Lag 1: F=  1.436  p=0.2333  
  Lag 2: F=  0.846  p=0.4320  
  Lag 3: F=  0.690  p=0.5603  

───────────────────────────────────────────────────────────────────────────
TEST: d_PCE_YOY → appreciation
  Lag 1: F=  0.086  p=0.7702  
  Lag 2: F=  0.641  p=0.5286  
  Lag 3: F=  0.955  p=0.4171  
TEST: appreciation → d_PCE_YOY
  Lag 1: F=  0.002  p=0.9652  
  La

In [7]:
print("\n" + "=" * 75)
print("GRANGER CAUSALITY SUMMARY — Best Lag per Direction")
print("=" * 75)
print(f"{'Direction':<45} {'Lag':>4} {'F-stat':>8} {'p-value':>9} {'Sig':>5}")
print("-" * 75)

for direction in results_df["direction"].unique():
    subset = results_df[results_df["direction"] == direction]
    best = subset.loc[subset["p_value"].idxmin()]
    print(f"{best['direction']:<45} {int(best['lag']):>4} "
          f"{best['F_stat']:>8.3f} {best['p_value']:>9.4f} {best['sig']:>5}")

print("=" * 75)
print("Significance: *** p<0.01  ** p<0.05  * p<0.10")


GRANGER CAUSALITY SUMMARY — Best Lag per Direction
Direction                                      Lag   F-stat   p-value   Sig
---------------------------------------------------------------------------
d_FEDFUNDS → appreciation                        1    0.210    0.6477      
appreciation → d_FEDFUNDS                        1    3.368    0.0692     *
UNRATE → appreciation                            1    0.859    0.3560      
appreciation → UNRATE                            1    1.436    0.2333      
d_PCE_YOY → appreciation                         3    0.955    0.4171      
appreciation → d_PCE_YOY                         2    0.070    0.9328      
d_MORTGAGE_SPREAD → appreciation                 1    0.309    0.5796      
appreciation → d_MORTGAGE_SPREAD                 6    1.315    0.2578      
Significance: *** p<0.01  ** p<0.05  * p<0.10


In [8]:
print("\n" + "=" * 80)
print("REGIME-CONDITIONAL GRANGER CAUSALITY")
print("=" * 80)

regime_order = [
    "Expansion / Normalization",
    "COVID Shock",
    "Inflation Surge",
    "Aggressive Tightening"
]

regime_results = []

for regime in regime_order:
    regime_data = granger_df[granger_df["regime_name"] == regime].copy()
    n_obs = len(regime_data)

    print(f"\n{'━' * 80}")
    print(f"REGIME: {regime} (n={n_obs} months)")
    print(f"{'━' * 80}")

    # Need enough observations for meaningful test
    # Rule of thumb: at least 3x the max lag
    if n_obs < 15:
        print(f"  ⚠ Too few observations for reliable Granger test (n={n_obs})")
        continue

    # Reduce max lag for smaller samples
    regime_max_lag = min(4, n_obs // 5)

    for macro_var in macro_cols:
        # Macro → Housing
        try:
            test = grangercausalitytests(
                regime_data[[appr_col, macro_var]].dropna(),
                maxlag=regime_max_lag, verbose=False
            )
            for lag in range(1, regime_max_lag + 1):
                f_stat = test[lag][0]["ssr_ftest"][0]
                p_val  = test[lag][0]["ssr_ftest"][1]
                sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.10 else ""
                regime_results.append({
                    "regime": regime, "direction": f"{macro_var} → {appr_col}",
                    "lag": lag, "F_stat": round(f_stat, 3),
                    "p_value": round(p_val, 4), "sig": sig
                })
        except Exception:
            pass

        # Housing → Macro
        try:
            test = grangercausalitytests(
                regime_data[[macro_var, appr_col]].dropna(),
                maxlag=regime_max_lag, verbose=False
            )
            for lag in range(1, regime_max_lag + 1):
                f_stat = test[lag][0]["ssr_ftest"][0]
                p_val  = test[lag][0]["ssr_ftest"][1]
                sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.10 else ""
                regime_results.append({
                    "regime": regime, "direction": f"{appr_col} → {macro_var}",
                    "lag": lag, "F_stat": round(f_stat, 3),
                    "p_value": round(p_val, 4), "sig": sig
                })
        except Exception:
            pass

regime_results_df = pd.DataFrame(regime_results)

# Print best result per regime and direction
print("\n" + "=" * 90)
print("REGIME-CONDITIONAL SUMMARY — Best Lag per Regime × Direction")
print("=" * 90)
print(f"{'Regime':<28} {'Direction':<35} {'Lag':>4} {'F':>7} {'p':>8} {'Sig':>4}")
print("-" * 90)

for regime in regime_order:
    r_sub = regime_results_df[regime_results_df["regime"] == regime]
    if len(r_sub) == 0:
        print(f"{regime:<28} Insufficient observations")
        continue
    for direction in r_sub["direction"].unique():
        d_sub = r_sub[r_sub["direction"] == direction]
        best = d_sub.loc[d_sub["p_value"].idxmin()]
        print(f"{regime:<28} {best['direction']:<35} {int(best['lag']):>4} "
              f"{best['F_stat']:>7.3f} {best['p_value']:>8.4f} {best['sig']:>4}")
    print()

print("=" * 90)


REGIME-CONDITIONAL GRANGER CAUSALITY

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGIME: Expansion / Normalization (n=45 months)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGIME: COVID Shock (n=12 months)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ⚠ Too few observations for reliable Granger test (n=12)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGIME: Inflation Surge (n=20 months)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGIME: Aggressive Tightening (n=37 months)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

REGIME-CONDITIONAL SUMMARY — Best Lag per Regime × Direction
Regime                    

In [9]:
# Load sentiment data
LM_PATH = "/Users/alicexu/Documents/MSAA/2026_Spring/5205/AML2_Final_Project/Dataset/Section 3- Sentiment Analysis/monthly_sentiment_lm.csv"
FB_PATH = "/Users/alicexu/Documents/MSAA/2026_Spring/5205/AML2_Final_Project/Dataset/Section 3- Sentiment Analysis/monthly_sentiment_finbert.csv"

# Load LM — check column names first
lm_sent = pd.read_csv(LM_PATH, parse_dates=["month"], index_col="month")
lm_sent = lm_sent.loc[:, ~lm_sent.columns.duplicated()]

fb_sent = pd.read_csv(FB_PATH, parse_dates=["month"], index_col="month")
fb_sent = fb_sent.loc[:, ~fb_sent.columns.duplicated()]

print("LM columns:", lm_sent.columns.tolist())
print("FB columns:", fb_sent.columns.tolist())

LM columns: ['positive_score', 'negative_score', 'uncertainty_score', 'net_tone']
FB columns: ['hawkish_score', 'dovish_score', 'neutral_score', 'finbert_negative_ratio']


In [10]:
# Merge sentiment into granger_df
# Adjust column names below based on the print output above
granger_sent = granger_df.copy()

granger_sent = granger_sent.merge(
    lm_sent, left_index=True, right_index=True, how="inner"
)
granger_sent = granger_sent.merge(
    fb_sent, left_index=True, right_index=True, how="inner"
)

print(f"Merged shape: {granger_sent.shape}")
print(f"Columns: {granger_sent.columns.tolist()}")

# Test: Does sentiment Granger-cause housing appreciation?
# Identify sentiment columns from the print above
# Replace these with your actual column names
lm_tone_col = "net_tone"        # adjust if different
fb_col = "finbert_negative_ratio"  # adjust if different

sentiment_cols = []
if lm_tone_col in granger_sent.columns:
    sentiment_cols.append(lm_tone_col)
if fb_col in granger_sent.columns:
    sentiment_cols.append(fb_col)

print("\n" + "=" * 75)
print("SENTIMENT → HOUSING APPRECIATION: Granger Causality")
print("=" * 75)

for sent_col in sentiment_cols:
    print(f"\nTEST: {sent_col} → {appr_col}")

    # Check stationarity of sentiment
    adf_p = adfuller(granger_sent[sent_col].dropna())[1]

    test_col = sent_col
    if adf_p >= 0.05:
        granger_sent[f"d_{sent_col}"] = granger_sent[sent_col].diff()
        test_col = f"d_{sent_col}"
        print(f"  (differenced: original p={adf_p:.4f})")

    try:
        test = grangercausalitytests(
            granger_sent[[appr_col, test_col]].dropna(),
            maxlag=max_lag, verbose=False
        )
        for lag in range(1, max_lag + 1):
            f_stat = test[lag][0]["ssr_ftest"][0]
            p_val  = test[lag][0]["ssr_ftest"][1]
            sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.10 else ""
            if lag <= 4:
                print(f"  Lag {lag}: F={f_stat:>7.3f}  p={p_val:.4f}  {sig}")
    except Exception as e:
        print(f"  Error: {e}")

print("=" * 75)

Merged shape: (114, 17)
Columns: ['appreciation', 'FEDFUNDS', 'UNRATE', 'PCE_YOY', 'MORTGAGE_SPREAD', 'regime_name', 'd_FEDFUNDS', 'd_PCE_YOY', 'd_MORTGAGE_SPREAD', 'positive_score', 'negative_score', 'uncertainty_score', 'net_tone', 'hawkish_score', 'dovish_score', 'neutral_score', 'finbert_negative_ratio']

SENTIMENT → HOUSING APPRECIATION: Granger Causality

TEST: net_tone → appreciation
  (differenced: original p=0.0899)
  Lag 1: F=  0.327  p=0.5688  
  Lag 2: F=  0.396  p=0.6739  
  Lag 3: F=  1.261  p=0.2918  
  Lag 4: F=  0.948  p=0.4397  

TEST: finbert_negative_ratio → appreciation
  Lag 1: F=  0.247  p=0.6202  
  Lag 2: F=  1.378  p=0.2565  
  Lag 3: F=  0.904  p=0.4420  
  Lag 4: F=  2.672  p=0.0363  **


In [11]:
from statsmodels.tsa.api import VAR

# Build VAR with appreciation, macro variables, and FinBERT
var_cols_with_sent = [appr_col, "d_FEDFUNDS", "UNRATE",
                      "d_PCE_YOY", "d_MORTGAGE_SPREAD",
                      "finbert_negative_ratio"]

var_data = granger_sent[var_cols_with_sent].dropna()

# Fit VAR and select lag order by AIC
var_model = VAR(var_data)
lag_selection = var_model.select_order(maxlags=6)
print(lag_selection.summary())

# Fit at optimal lag
optimal_lag = lag_selection.aic
print(f"\nOptimal lag by AIC: {optimal_lag}")

var_result = var_model.fit(optimal_lag)

# Granger causality within the VAR framework
print("\n" + "=" * 65)
print(f"VAR Granger Test: finbert_negative_ratio → {appr_col}")
print("(controlling for all macro variables)")
print("=" * 65)

test = var_result.test_causality(
    appr_col,
    ["finbert_negative_ratio"],
    kind="f"
)
print(f"  F-stat: {test.test_statistic:.3f}")
print(f"  p-value: {test.pvalue:.4f}")
sig = "***" if test.pvalue < 0.01 else "**" if test.pvalue < 0.05 else "*" if test.pvalue < 0.10 else ""
print(f"  Significance: {sig if sig else 'Not significant'}")

# Also test the reverse: does housing predict stress?
print(f"\nVAR Granger Test: {appr_col} → finbert_negative_ratio")
test_rev = var_result.test_causality(
    "finbert_negative_ratio",
    [appr_col],
    kind="f"
)
print(f"  F-stat: {test_rev.test_statistic:.3f}")
print(f"  p-value: {test_rev.pvalue:.4f}")
sig_rev = "***" if test_rev.pvalue < 0.01 else "**" if test_rev.pvalue < 0.05 else "*" if test_rev.pvalue < 0.10 else ""
print(f"  Significance: {sig_rev if sig_rev else 'Not significant'}")

print("=" * 65)

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0      -1.838      -1.689      0.1591      -1.778
1      -4.303     -3.260*     0.01355     -3.880*
2     -4.645*      -2.708   0.009673*      -3.860
3      -4.307      -1.476     0.01378      -3.159
4      -4.191     -0.4660     0.01592      -2.681
5      -4.047      0.5723     0.01931      -2.174
6      -3.616       1.897     0.03196      -1.381
-------------------------------------------------

Optimal lag by AIC: 2

VAR Granger Test: finbert_negative_ratio → appreciation
(controlling for all macro variables)
  F-stat: 1.394
  p-value: 0.2489
  Significance: Not significant

VAR Granger Test: appreciation → finbert_negative_ratio
  F-stat: 0.048
  p-value: 0.9535
  Significance: Not significant


In [12]:
# Test within Aggressive Tightening (largest subsample after Expansion)
for regime in ["Expansion / Normalization", "Aggressive Tightening"]:
    regime_data = granger_sent[granger_sent["regime_name"] == regime].copy()

    regime_var_data = regime_data[
        [appr_col, "d_FEDFUNDS", "UNRATE",
         "d_PCE_YOY", "d_MORTGAGE_SPREAD", "finbert_negative_ratio"]
    ].dropna()

    n = len(regime_var_data)
    print(f"\n{'=' * 65}")
    print(f"REGIME: {regime} (n={n})")
    print(f"{'=' * 65}")

    if n < 20:
        print("  Too few observations for VAR")
        continue

    try:
        regime_var = VAR(regime_var_data)
        # Use lag 1 or 2 for smaller samples
        regime_lag = min(2, n // 10)
        regime_fit = regime_var.fit(regime_lag)

        test = regime_fit.test_causality(
            appr_col,
            ["finbert_negative_ratio"],
            kind="f"
        )
        sig = "***" if test.pvalue < 0.01 else "**" if test.pvalue < 0.05 else "*" if test.pvalue < 0.10 else ""
        print(f"  FinBERT → {appr_col}:  F={test.test_statistic:.3f}  "
              f"p={test.pvalue:.4f}  {sig}")

        test_rev = regime_fit.test_causality(
            "finbert_negative_ratio",
            [appr_col],
            kind="f"
        )
        sig_rev = "***" if test_rev.pvalue < 0.01 else "**" if test_rev.pvalue < 0.05 else "*" if test_rev.pvalue < 0.10 else ""
        print(f"  {appr_col} → FinBERT:  F={test_rev.test_statistic:.3f}  "
              f"p={test_rev.pvalue:.4f}  {sig_rev}")
    except Exception as e:
        print(f"  Error: {e}")


REGIME: Expansion / Normalization (n=45)
  FinBERT → appreciation:  F=5.875  p=0.0034  ***
  appreciation → FinBERT:  F=0.194  p=0.8239  

REGIME: Aggressive Tightening (n=37)
  FinBERT → appreciation:  F=1.921  p=0.1506  
  appreciation → FinBERT:  F=0.080  p=0.9229  
